# Scam checker: look at the data (and the results)

Read-only: this notebook never calls a model and never changes a file.

1. `python -m scamcheck build` first (it merges `data/synthetic`, `data/cases` and `data/real` into `data/final`).
2. Pick a Python kernel that has `pandas` (`pip install -r requirements.txt`).
3. Run all cells. Section 9 shows the scores once you have run an evaluation on Modal.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from scamcheck import messages, target  # noqa: E402
from scamcheck.build import key  # noqa: E402
from scamcheck.policy import flags_in  # noqa: E402

FINAL = ROOT / "data" / "final"
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

frames = [pd.DataFrame([json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]).assign(split=p.stem)
          for p in sorted(FINAL.glob("*.jsonl")) if p.stat().st_size]
if not frames:
    raise SystemExit(f"Nothing in {FINAL}: run `python -m scamcheck build` first.")
df = pd.concat(frames, ignore_index=True)
df["chars"] = df["text"].str.len()
print(f"{len(df):,} messages in {FINAL}")

## 1. What `build` read and wrote

In [ ]:
json.loads((FINAL / "stats.json").read_text(encoding="utf-8"))

## 2. Where the rows came from (synthetic, your cases, real)

In [ ]:
pd.crosstab(df["origin"], df["split"], margins=True)

## 3. Size of each split and the share of scams

Aim for roughly 40-50% scams in training.

In [ ]:
df.groupby("split").agg(messages=("text", "size"), scam_share=("scam", "mean")).round(2)

## 4. Scam types per split

Every type needs a few hundred training rows; a thin type will be weak.

In [ ]:
pd.crosstab(df["type"], df["split"], margins=True)

## 5. Channels, senders and links

Watch for shortcuts: if only scams have links or long texts, the model learns the shortcut instead of the scam.

In [ ]:
display(pd.crosstab(df["channel"].replace("", "(none)"), df["scam"], margins=True))
summary = df.assign(has_sender=df["sender"].fillna("") != "", has_link=df["text"].str.contains("http"))
summary.groupby("scam")[["has_sender", "has_link"]].mean().round(2)

## 6. Message length (characters)

In [ ]:
df.groupby(["split", "scam"])["chars"].describe()[["count", "min", "25%", "50%", "75%", "max"]].round(0)

## 7. Checks that must pass

- every red flag occurs in its message;
- no message appears in more than one split (a leak inflates the scores).

In [ ]:
scams = df[df["scam"]]
bad_flags = scams[[flags_in(t, list(f)) != list(f) for t, f in zip(scams["text"], scams["red_flags"], strict=True)]]
splits_per_text = df.assign(k=df["text"].map(key)).groupby("k")["split"].nunique()
print(f"red flags not in the message: {len(bad_flags)}")
print(f"messages found in more than one split: {(splits_per_text > 1).sum()}")

## 8. Read some examples

Change `TYPE` to look at another kind (`none` for genuine).

In [ ]:
TYPE = "refund"
cols = ["split", "origin", "channel", "sender", "text", "red_flags", "advice"]
pick = df[df["type"] == TYPE]
pick.sample(min(8, len(pick)), random_state=1)[cols]

In [ ]:
# genuine messages that look like a scam: the hard negatives
look = df[df["target"].str.startswith("lookalike:")]
look.sample(min(8, len(look)), random_state=2)[["target", "channel", "sender", "text"]]

In [ ]:
# exactly what the tuned model is trained on: the short prompt, and the answer it learns
row = df[df["split"] == "train"].iloc[0].to_dict()
for m in messages(row, short=True):
    print(f"--- {m['role']} ---\n{m['content']}\n")
print("--- answer (training target) ---")
print(target(row))

## 9. Results (after `modal run modal_train.py::evaluate ...`)

The scores of every model evaluated so far, and the mistakes of one of them.

In [ ]:
results = {p.stem: json.loads(p.read_text(encoding="utf-8")) for p in sorted((ROOT / "results").glob("*.json"))
           if "scores" in p.read_text(encoding="utf-8")[:2000]}
if not results:
    print("No results yet.")
else:
    rows = [{"model": r["model"], "test set": name, **{k: v for k, v in s.items() if k != "confusion"}}
            for r in results.values() for name, s in r["scores"].items()]
    display(pd.DataFrame(rows).round(3))

In [ ]:
# the mistakes of one model: missed scams and false alarms
RESULT, SET = (next(iter(results)) if results else None), "test"
if RESULT:
    p = pd.DataFrame(results[RESULT]["predictions"][SET])
    display(p[p["scam"] & ~p["pred_scam"]][["type", "text"]].head(15))   # missed scams
    display(p[~p["scam"] & p["pred_scam"]][["pred_type", "text"]].head(15))  # false alarms

In [ ]:
# the training loss curve
for p in sorted((ROOT / "results").glob("train_*.json")):
    log = json.loads(p.read_text())
    ax = pd.DataFrame(log["loss"]).plot(x="step", y="loss", title=p.stem)
    pd.DataFrame(log["val_loss"]).plot(x="step", y="loss", ax=ax, label="val")